# 01. Exploratory Data Analysis: Predicting TB Drug Resistance from Genomic SNPs

**Project:** Genomic Prediction of *M. tuberculosis* Drug Resistance  
**Course:** Bioinformatics Unit for Computer Science  
**Objective:** Investigate whether single-nucleotide polymorphisms (SNPs) in known resistance genes can predict phenotypic drug resistance across four antibiotics in *M. tuberculosis*.

---

### Problem Formulation
We are framing this as a binary classification task:
- **Feature Matrix ($X$):** Sparse binary matrix where each row is an isolate (patient bacterial sample) and each column indicates whether a specific mutation in our target gene panel is present (`1`) or absent (`0`).
- **Labels ($y$):** Laboratory culture susceptibility tests (`1` = Resistant, `0` = Susceptible).
- **Target Drugs & Genes:**
  - First-line: **Rifampicin** (`rpoB`), **Isoniazid** (`katG`, `inhA`, `fabG1` promoter)
  - Second-line: **Levofloxacin** (`gyrA`), **Moxifloxacin** (`gyrA`)

### What we need to answer in this EDA before modeling:
1. **Class balance & sample size:** How many samples do we have per drug, and what is the label balance ($P(y=1)$)?
2. **Co-resistance & multi-drug resistance (MDR):** How correlated are resistance labels between drugs? If an isolate has both RIF and INH resistance, could a model exploit cross-drug correlations rather than learning causal biology?
3. **Mutation enrichment:** Which individual SNPs show the strongest association with resistance?
4. **Phylogenetic confounding:** Are there mutations that look predictive but are actually just lineage markers (ancestry)? How do resistance rates vary across lineages, and what does this mean for our cross-validation split strategy?
5. **Data quality & sparsity:** How sparse is our feature matrix, and are there missing/null calls to handle?

In [ ]:
# Step 1: Import standard scientific python libraries and our project helpers
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tb_resistance.eda import (
    load_drug_dataset,
    summarize_drug_cohorts,
    top_mutations_by_resistance,
    lineage_distribution,
    gene_burden_analysis,
    multidrug_overlap,
)

# Configure clean plots
%matplotlib inline
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.dpi'] = 120

# Processed data directory containing our per-drug matrices
DATA_DIR = Path('../data/processed')
print('Loaded libraries and pointing to processed data at:', DATA_DIR.resolve())

## 1. Dataset Scale & Label Balance

First, let's load the summary of all four drug datasets. We want to check our sample sizes, the split between resistant ($y=1$) and susceptible ($y=0$) isolates, and the quality of the laboratory phenotype calls.

In [ ]:
cohort_summary = summarize_drug_cohorts(DATA_DIR)
cohort_summary

In [ ]:
# Plot sample counts and class balance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(cohort_summary))
width = 0.35

# Left plot: Raw counts of Resistant vs Susceptible
ax1.bar(x - width/2, cohort_summary['resistant'], width, label='Resistant (1)', color='#c93b2b', alpha=0.9)
ax1.bar(x + width/2, cohort_summary['susceptible'], width, label='Susceptible (0)', color='#4a7c59', alpha=0.9)
ax1.set_ylabel('Number of Isolates')
ax1.set_title('Cohort Size & Class Breakdown')
ax1.set_xticks(x)
ax1.set_xticklabels([d.capitalize() for d in cohort_summary.index])
ax1.legend()

# Right plot: Resistance prevalence (% resistant)
sns.barplot(data=cohort_summary.reset_index(), x='drug', y='prevalence_pct', ax=ax2, hue='drug', palette='mako', legend=False)
ax2.set_ylabel('Resistant Isolates (%)')
ax2.set_title('Resistance Prevalence Rate')
ax2.set_xticks(range(len(cohort_summary)))
ax2.set_xticklabels([d.capitalize() for d in cohort_summary.index])
for i, val in enumerate(cohort_summary['prevalence_pct']):
    ax2.text(i, val + 1, f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Our Observations on Label Balance:
- **Sample sizes:** We have ~11,500 to 12,300 isolates per drug—plenty of training data for tabular models.
- **First-line drugs (RIF & INH):** Moderate class balance (~39.3% resistant for Rifampicin, ~46.4% for Isoniazid). Standard binary cross-entropy / logistic loss will work fine without heavy class weighting.
- **Second-line fluoroquinolones (LEV & MXF):** Noticeably more imbalanced (~17.7% and ~14.2% resistant). Raw classification accuracy will be misleading here—a trivial all-zero predictor would get 86% accuracy. When we evaluate LEV and MXF models, we need to focus on **PR-AUC (Precision-Recall AUC)**, **F1-score**, and **Sensitivity/Recall**.
- **Data Quality:** Almost all calls (>95%) are day 14 readings with HIGH phenotype quality.

## 2. Multi-Drug Resistance (MDR) & Label Correlation

In clinical regimens, Rifampicin and Isoniazid are prescribed together. Bacteria that evolve resistance to both are termed **Multi-Drug Resistant (MDR-TB)**.

From a data science standpoint, this creates a major **multicollinearity / shortcut learning risk**:  
If an isolate is resistant to both RIF and INH, its genome will carry mutations in *both* `rpoB` (RIF target) and `katG`/`inhA` (INH targets). If we pass the whole panel to a model predicting Rifampicin, the model could easily cheat by learning `katG` mutations rather than true causal `rpoB` mutations.

Let's compute the pairwise co-resistance matrix to see how prevalent MDR-TB is in our data.

In [ ]:
# Compute shared isolates and co-resistant counts
shared, both_res = multidrug_overlap(DATA_DIR)
print('--- Number of Shared Isolates Tested for Both Drugs ---')
display(shared)
print('\n--- Number of Isolates Resistant to BOTH Drugs (Co-Resistant) ---')
display(both_res)

In [ ]:
# Visualize co-resistance rate as a heatmap
concordance = (both_res / shared * 100.0).astype(float)
plt.figure(figsize=(7, 5))
sns.heatmap(concordance, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': '% Co-Resistant'})
plt.title('Pairwise Co-Resistance Concordance (%)')
plt.show()

### Our Observations on Co-Resistance:
- **High MDR prevalence:** Out of 11,371 isolates tested for both first-line drugs, **4,191 isolates are resistant to both RIF and INH** (a 36.9% co-resistance rate).
- **Fluoroquinolone pairing:** Levofloxacin and Moxifloxacin show 13.9% co-resistance (1,702 isolates), reflecting cross-resistance within the same fluoroquinolone class.
- **Modeling decision:** To test whether our models are learning genuine biological resistance mechanisms rather than co-resistance shortcuts, we should benchmark two feature sets:
  1. *Target-gene only* (e.g., using only `rpoB` for Rifampicin).
  2. *Full panel* (using all 5 genes).

## 3. Gene-Level Mutation Burden

Before looking at individual SNPs, let's test a simple baseline biological hypothesis:  
**Do resistant isolates actually carry mutations in their expected target genes, while susceptible isolates don't?**

In [ ]:
ds_rif = load_drug_dataset(DATA_DIR, 'rifampicin')
print('=== RIFAMPICIN: Mutation Burden across Genes ===')
burden_rif = gene_burden_analysis(ds_rif.features, ds_rif.labels['label'], ds_rif.metadata)
display(burden_rif)

ds_inh = load_drug_dataset(DATA_DIR, 'isoniazid')
print('\n=== ISONIAZID: Mutation Burden across Genes ===')
burden_inh = gene_burden_analysis(ds_inh.features, ds_inh.labels['label'], ds_inh.metadata)
display(burden_inh)

### Our Observations on Gene Burden:
- **Strong biological signal:** For Rifampicin, **96.6%** of resistant isolates carry $\ge 1$ mutation in `rpoB`, compared to only 14.8% of susceptible ones ($\Delta = 81.8\%$). For Isoniazid, **91.2%** of resistant isolates carry $\ge 1$ mutation in `katG`, compared to 44.8% of susceptible ones.\n- **Why are there mutations in susceptible isolates?** Notice that in `katG`, 44.8% of susceptible isolates have a mutation. Why would a susceptible isolate have a mutation in a resistance gene? We explore this in Section 5—it turns out to be **phylogenetic lineage markers** rather than resistance-conferring mutations.

## 4. Top Resistance-Associated Mutations (The Hotspot SNPs)

Now let's drill down into individual mutations. We rank all panel mutations by their frequency difference between resistant and susceptible isolates ($\Delta\% = \%_{Resistant} - \%_{Susceptible}$) and calculate their **Odds Ratio**.

In [ ]:
# Extract top mutations in each drug's primary target gene
drugs_and_genes = [
    ('rifampicin', 'rpoB'),
    ('isoniazid', 'katG'),
    ('levofloxacin', 'gyrA'),
    ('moxifloxacin', 'gyrA')
]

for drug, gene in drugs_and_genes:
    ds = load_drug_dataset(DATA_DIR, drug)
    top = top_mutations_by_resistance(ds.features, ds.labels['label'], ds.metadata, gene=gene, top_n=6)
    print(f'\n=== TOP {gene.upper()} MUTATIONS FOR {drug.upper()} ===')
    display(top[['gene', 'mutation', 'resistant_count', 'resistant_freq_pct', 'susceptible_freq_pct', 'delta_pct', 'odds_ratio']])

In [ ]:
# Bar plots comparing mutation frequencies between classes
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, (drug, gene) in enumerate(drugs_and_genes):
    ds = load_drug_dataset(DATA_DIR, drug)
    top = top_mutations_by_resistance(ds.features, ds.labels['label'], ds.metadata, gene=gene, top_n=6)
    df_p = top.reset_index()
    ax = axes[idx]
    y_idx = np.arange(len(df_p))
    
    ax.barh(y_idx - 0.18, df_p['resistant_freq_pct'], height=0.36, color='#c93b2b', label='Resistant %' if idx==0 else '')
    ax.barh(y_idx + 0.18, df_p['susceptible_freq_pct'], height=0.36, color='#4a7c59', label='Susceptible %' if idx==0 else '')
    ax.set_yticks(y_idx)
    ax.set_yticklabels(df_p['mutation'])
    ax.invert_yaxis()
    ax.set_xlabel('Frequency (%)')
    ax.set_title(f'{drug.capitalize()} ({gene})')
    if idx == 0:
        ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

### Our Observations on Top Mutations:
- **Clear hotspot drivers:**
  - **Rifampicin:** `rpoB@S450L` is the standout driver: present in **62.1%** of resistant isolates vs just **1.5%** of susceptible isolates (Odds Ratio = **104.8**!). Secondaries include `rpoB@D435V` (10.7%, OR = 31.1) and `rpoB@H445D` (4.4%, OR = 45.3).
  - **Isoniazid:** `katG@S315T` is essentially a deterministic indicator: present in **86.4%** of resistant isolates vs **1.2%** of susceptible isolates (Odds Ratio = **500.9**!).
  - **Fluoroquinolones:** `gyrA@D94G` (35.0%, OR = 50.1) and `gyrA@A90V` (21.7%, OR = 39.4) drive most resistance.
- **Takeaway for our models:** Because a few dominant hotspot SNPs account for the vast majority of resistance, a sparse linear model like L1-penalized Logistic Regression should perform very well and give us interpretable coefficients.

## 5. Investigating Phylogenetic Confounders: The Lineage Trap

In bacteria, clones diverge into distinct evolutionary lineages (e.g., Lineage 2 East-Asian/Beijing, Lineage 4 Euro-American). This creates two challenges we have to solve in our ML pipeline:

1. **Phylogenetic Markers vs. Causal Mutations:** Look closely at `katG@R463L` and `rpoB@A1075A` in our feature list. They appear in ~44% of resistant isolates, but also in ~44% of susceptible isolates! Why? They do not confer drug resistance at all—they are lineage-defining SNPs fixed in Lineages 2 and 4. A naive model might assign them non-zero weights.
2. **Resistance Clumping by Lineage:** Certain lineages have experienced major MDR outbreaks. Let's see if resistance rates vary by lineage.

In [ ]:
ds_rif = load_drug_dataset(DATA_DIR, 'rifampicin')
lin_df = lineage_distribution(ds_rif.labels, ds_rif.lineage)
lin_df

In [ ]:
major_lin = lin_df[lin_df['total_isolates'] >= 50].reset_index()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Lineage proportion
sns.barplot(data=major_lin, x='lineage', y='share_of_cohort_pct', ax=ax1, hue='lineage', palette='crest', legend=False)
ax1.set_title('Lineage Share in Cohort (%)')
ax1.set_ylabel('% of Isolates')
ax1.set_xticks(range(len(major_lin)))
ax1.set_xticklabels(major_lin['lineage'], rotation=25)

# Resistance rate per lineage
sns.barplot(data=major_lin, x='lineage', y='prevalence_pct', ax=ax2, hue='lineage', palette='flare', legend=False)
ax2.set_title('Rifampicin Resistance Rate by Lineage (%)')
ax2.set_ylabel('Prevalence (% Resistant)')
ax2.set_xticks(range(len(major_lin)))
ax2.set_xticklabels(major_lin['lineage'], rotation=25)

plt.tight_layout()
plt.show()

### Our Observations on Lineage:
- **Major imbalance in resistance across lineages:**
  - **Lineage 2 (Beijing):** **50.8%** of isolates are resistant to Rifampicin.
  - **Lineage 4 (Euro-American):** **32.1%** resistant.
  - **Lineage 1 (Indo-Oceanic):** Only **21.7%** resistant.
- **The Data Leakage Hazard:** If we use standard random `train_test_split()`, isolates from the exact same clonal branch will end up in both training and test folds. Our models could achieve artificially inflated test scores by memorizing lineage markers instead of learning real resistance mutations.
- **Our Strategy:** We must implement **Lineage-stratified cross-validation** to test how well our models generalize across distinct bacterial lineages.

## 6. Feature Sparsity & Data Quality Check

Our feature matrix has 6,526 columns across the 5 panel genes. How many of them are actually populated? And how often are calls missing (null or filter-failed) or heterozygous?

In [ ]:
ds_rif_q = load_drug_dataset(DATA_DIR, 'rifampicin', load_quality_matrices=True)
n_missing_isolates = (ds_rif_q.missing.sum(axis=1) > 0).sum()
n_het_isolates = (ds_rif_q.het.sum(axis=1) > 0).sum()

print(f'Total Rifampicin Isolates: {len(ds_rif_q.labels):,}')
print(f'Isolates with >=1 missing/null feature call: {n_missing_isolates:,} ({n_missing_isolates / len(ds_rif_q.labels) * 100:.2f}%)')
print(f'Isolates with >=1 heterozygous call: {n_het_isolates:,} ({n_het_isolates / len(ds_rif_q.labels) * 100:.2f}%)')

### Our Observations on Sparsity & Quality:
- **High sparsity:** Out of 6,526 total features, roughly ~5,000 are all-zero within each drug cohort (they are ultra-rare variants observed in only 1 or 2 isolates across the entire worldwide dataset).
- **Clean calls:** Only **5.67%** of isolates have any missing feature flag in the entire panel, and heteroresistance is essentially zero.
- **Preprocessing plan:** Before model fitting, we should filter out ultra-rare features (e.g. minimum frequency threshold $\ge 3$) **fitted strictly on the training folds** to avoid data leakage. This will drop our feature dimension from 6,500 down to a few hundred informative columns.

## 7. Our Modeling Plan & Engineering Next Steps

Based on this EDA, here is the concrete strategy for our next modeling script:

| Finding from our EDA | Challenge for ML | How We Will Address It in Code |
| :--- | :--- | :--- |
| **High dimensionality & rare variants** | 6,526 features, mostly zeros; risk of overfitting. | Feature filter (count $\ge 3$) wrapped inside a scikit-learn Pipeline / fitted strictly on training folds. |
| **Dominant single mutations** | `rpoB@S450L`, `katG@S315T` have massive odds ratios (>100). | Baseline with **L1/L2 penalized Logistic Regression** (sparse, interpretable weights) and **Random Forest** (captures non-linear combinations). |
| **Lineage confounding** | Resistance prevalence differs across lineages (50.8% in L2 vs 21.7% in L1); lineage markers exist. | Evaluate with **Lineage-stratified K-Fold CV** to evaluate true out-of-lineage generalization. |
| **MDR-TB co-resistance** | 4,191 isolates resistant to both RIF and INH. | Compare models trained on **target-gene-only features** (e.g. `rpoB` for RIF) vs **all-panel features** to check for shortcut learning. |
| **Class imbalance in Fluoroquinolones** | Only ~14-18% resistant in LEV/MXF. | Evaluate using **PR-AUC, ROC-AUC, Sensitivity, and Specificity**, rather than raw accuracy. |